# Nghiên cứu so sánh các phương pháp xử lý ảnh cho ANPR (`image_processing_study`)

Đồ án môn Xử lý ảnh: so sánh ~13 phương pháp xử lý ảnh (bám khung chương trình môn học) áp dụng cho
bài toán đọc biển số, **trên cùng một kiến trúc model nhỏ train from scratch** (CRNN + CTC, không
dùng PARSeq vì PARSeq pretrained đã quá mạnh nên sẽ san phẳng khác biệt giữa các cách xử lý ảnh).

Hai thí nghiệm:
- **Thí nghiệm A** -- train riêng 1 model CRNN cho MỖI phương pháp xử lý ảnh (cùng kiến trúc/seed/
  hyperparameter, cùng 1 bộ train/val/test split cố định), so sánh exact-match accuracy / CER.
- **Thí nghiệm B** -- tạo suy giảm tổng hợp có biết trước (blur/noise) trên đúng tập test của
  Thí nghiệm A, đo PSNR/SSIM và OCR accuracy (qua model `raw` đã train ở Thí nghiệm A) của các
  phương pháp phục hồi ảnh, gồm cả agent RL deblur (`rl_deblur/`, deep-learning) để so với các
  phương pháp cổ điển.

**Cần chuẩn bị trước khi chạy** (trên máy local, trong thư mục repo `parseq_official_pipeline`):

```bash
zip -r parseq_ip_study_data.zip \
  image_processing_study color_filtered rl_deblur \
  outputs/rl_deblur/checkpoints/best_deblur_agent.pt \
  -x "*/__pycache__/*" "*.ipynb"
```

Zip này gộp cả **code** (`image_processing_study/`, `rl_deblur/` -- import thẳng, không cần
`%%writefile` lại trong notebook) lẫn **data** (`color_filtered/`, checkpoint RL đã train nếu có
-- tùy chọn, thiếu thì notebook tự bỏ qua đúng `rl_deblur_restore`).

Rồi upload file `parseq_ip_study_data.zip` lên Google Drive (vd thư mục gốc `My Drive/`). Mỗi lần
sửa code ở local, zip + upload lại là xong -- không cần sửa gì trong notebook.

**Runtime**: Menu `Runtime > Change runtime type > GPU (T4)`.

## 1. Mount Google Drive + cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sua duong dan nay cho khop voi noi ban upload file zip tren Drive
DRIVE_ZIP_PATH = '/content/drive/MyDrive/parseq_ip_study_data.zip'
PROJECT_DIR = '/content/project'

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Giải nén dữ liệu + code, cài dependency

In [ ]:
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
!unzip -q -o "{DRIVE_ZIP_PATH}" -d {PROJECT_DIR}
!ls {PROJECT_DIR}

In [ ]:
%pip install -q opencv-python-headless scikit-image PyWavelets pandas pillow tqdm matplotlib

In [ ]:
import os, sys
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('cwd:', os.getcwd())

import image_processing_study
print('image_processing_study package loaded from:', image_processing_study.__file__)

RL_CHECKPOINT = f'{PROJECT_DIR}/outputs/rl_deblur/checkpoints/best_deblur_agent.pt'
INCLUDE_RL = os.path.exists(RL_CHECKPOINT)
print('rl_deblur checkpoint found:' if INCLUDE_RL else 'rl_deblur checkpoint NOT found (rl_deblur_restore will be skipped):', RL_CHECKPOINT)

# rl_deblur_restore la 1 Method duoc nap dong (khong nam trong CORE_METHODS), nen cac ham ve
# hinh (make_method_preview_grid / make_restoration_grid) can duoc truyen no rieng qua
# extra_methods= de xuat hien trong hinh -- neu khong no van duoc train/danh gia binh thuong
# o Thi nghiem A/B (qua build_registry), chi la khong len hinh minh hoa.
from image_processing_study.methods import try_build_rl_deblur_method

RL_METHOD_FOR_VIZ = try_build_rl_deblur_method(RL_CHECKPOINT, device='cpu') if INCLUDE_RL else None
EXTRA_METHODS_FOR_VIZ = [RL_METHOD_FOR_VIZ] if RL_METHOD_FOR_VIZ is not None else None

## 3. Xem trước ảnh sau xử lý (trước khi train)

Với mỗi trong 12 phương pháp cốt lõi + `rl_deblur_restore` (nếu có checkpoint), in ra 5 ảnh mẫu
khác nhau sau khi xử lý -- đúng những gì model sẽ thấy khi train. Kiểm tra bằng mắt trước khi bỏ
thời gian train cả 13 model.

In [ ]:
from image_processing_study.visualize import make_method_preview_grid
from IPython.display import Image as IPImage, display

preview_path = make_method_preview_grid(
    'outputs/image_processing_study/samples/preview_processed_samples.png',
    num_samples=5,
    extra_methods=EXTRA_METHODS_FOR_VIZ,
)
display(IPImage(str(preview_path)))

## 4. Thí nghiệm A -- train 1 CRNN riêng cho mỗi phương pháp xử lý ảnh

Dataset nhỏ (~2944 ảnh train sau split 80/10/10) + model nhẹ (~2M tham số). Tối đa `epochs=100`
nhưng có **early stopping**: nếu `val_exact_acc` không cải thiện sau `patience=10` epoch liên
tiếp thì dừng sớm (log dòng `[method] early stopping at epoch ...`) -- tránh train đủ 100 epoch
cho những method đã bão hòa từ sớm, tổng thời gian cho 13 method vẫn chỉ khoảng vài chục phút
trên GPU T4.

`num_workers=0` (không dùng multiprocessing DataLoader workers) -- dataset nhỏ nên không cần, và
`num_workers>0` hay gây lỗi/nhiễu `AssertionError: can only test a child process` trong kernel
Colab/Jupyter khi DataLoader worker bị garbage-collect.

In [ ]:
from image_processing_study.run_experiment_a import run_sweep
import argparse

args = argparse.Namespace(
    output_dir='outputs/image_processing_study/experiment_a',
    methods=None,          # None = chay full 13 method (hoac 12 neu khong co rl checkpoint)
    include_rl=INCLUDE_RL,
    rl_checkpoint=RL_CHECKPOINT,
    epochs=100,
    patience=10,
    batch_size=64,
    num_workers=0,
    lr=1e-3,
    seed=42,
    split_seed=42,
    limit_train=None,
    limit_val=None,
    limit_test=None,
    device='cuda',
)
results_a = run_sweep(args)
results_a

## 5. Thí nghiệm B -- phục hồi ảnh suy giảm tổng hợp (PSNR/SSIM + OCR)

In [ ]:
from image_processing_study.run_experiment_b import run_evaluation

args_b = argparse.Namespace(
    output_dir='outputs/image_processing_study/experiment_b',
    raw_checkpoint='outputs/image_processing_study/experiment_a/raw/best_model.pt',
    include_rl=INCLUDE_RL,
    rl_checkpoint=RL_CHECKPOINT,
    degrade_seed=123,
    split_seed=42,
    limit=None,
    device='cuda',
)
results_b = run_evaluation(args_b)
results_b

## 6. Ảnh minh họa + biểu đồ so sánh

In [ ]:
from image_processing_study.visualize import make_method_grid, make_restoration_grid, plot_experiment_a_bar_chart, plot_experiment_b_bar_chart
from image_processing_study.dataset import build_split

sample_path = str(build_split(seed=42)['test'][0][0])
make_method_grid(sample_path, 'outputs/image_processing_study/samples/experiment_a_methods_grid.png', extra_methods=EXTRA_METHODS_FOR_VIZ)
make_restoration_grid('outputs/image_processing_study/samples/experiment_b_restoration_grid.png', extra_methods=EXTRA_METHODS_FOR_VIZ)
plot_experiment_a_bar_chart('outputs/image_processing_study/experiment_a/comparison.csv', 'outputs/image_processing_study/samples/experiment_a_accuracy_bar.png')
plot_experiment_b_bar_chart('outputs/image_processing_study/experiment_b/comparison.csv', 'outputs/image_processing_study/samples/experiment_b_psnr_bar.png')

from IPython.display import Image as IPImage, display
for p in ['experiment_a_methods_grid.png', 'experiment_a_accuracy_bar.png', 'experiment_b_restoration_grid.png', 'experiment_b_psnr_bar.png']:
    display(IPImage(f'outputs/image_processing_study/samples/{p}'))

## 7. Lưu kết quả về Google Drive

Colab runtime là tạm thời (mất hết khi ngắt kết nối) -- nén toàn bộ `outputs/image_processing_study/`
(checkpoint mỗi method, history, predictions, 2 bảng so sánh, ảnh minh họa) và copy về Drive.

In [ ]:
!zip -q -r outputs_image_processing_study_result.zip outputs/image_processing_study
!cp outputs_image_processing_study_result.zip /content/drive/MyDrive/
print('Da luu outputs_image_processing_study_result.zip vao Google Drive (My Drive).')